# Lab: SVMs
## CMSE 381 - Spring 2022
## April 15, 2022

![](https://upload.wikimedia.org/wikipedia/commons/thumb/7/72/SVM_margin.png/300px-SVM_margin.png)

In [ ]:
# Everyone's favorite standard imports
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
%matplotlib inline
import time


# ML imports we've used previously
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error



In this module we are going to test out the SVM methods discussed in class. The books definitions as we talked about in class are:

- *Maximal Margin Classifiers*, where the goal was to find a separating hyperplane with no misclassifications, 
- *Support vector classifiers*, where we allow for a soft margin and hence some correct issues, but only allow for a linear kernal, and 
- *Support vecotr machines*, where we have a soft margin and an option for kernels. 

It turns out that `sklearn` has only one function to do all of this, and we can basically trick it (by understanding the innards) into doing any of them.  **<font color=red>However, there are two things that will likely be confusing. </font>**
- The command is just called `SVC`, but you should thinking of it as doing the most general SVM as defined in the book and then we can modify our inputs to allow for the other options as necessary.
- The cost input parameter is not the same as the `C` defined in the book. 

For now, we're going to mess with some synthetic data (meaning I generated it and saved it as a CSV for you). 

In [ ]:
data = np.loadtxt('SVM-Data.csv')

X = data[:,:2]
y = data[:,2]

In [ ]:
plt.scatter(X[:,0], X[:,1], s=70, c=y, cmap=mpl.cm.Paired)
plt.xlabel('X1')
plt.ylabel('X2')

And then, tada! Here's all it takes to fit your support vector machine. 

In [ ]:
from sklearn.svm import SVC


In [ ]:
svc = SVC(C=1, kernel='linear', )
svc.fit(X, y)

&#9989; **<font color=red>Do this:</font>** Use your trained model to figure out the equation of the hyperplane. Plot it on a graph with the data points.  Does the resulting hyperplane seem reasonable? 

In [ ]:
# Your code here 


Remember that the SVM setting only uses a subset of observations, called *support vectors* to actually determine this hyperplane. The `svc` object keeps track of those for us. 

In [ ]:
# Here are the indices of the support vectors from our original X matrix

svc.support_

In [ ]:
# It also keeps track of the points themselves 
svc.support_vectors_

&#9989; **<font color=red>Do this:</font>** Draw a scatter plot of these points on top of the drawing you've already been building.  Do these points make sense to be the support vectors?

In [ ]:
# Your code here 


Now that you have a sense of what's going on in the svc function, I've built you a function that will make this nice drawing for us without much effort.  We can hand it our `X` and `y` data, along with the trained `svc` to get the plot, with some added stuff, including dashed lines for the margin and colors for which side of the hyperplane you're on (blue for -1, red for +1).

In [ ]:
# Run this cell to define the function
def plot_svc(svc, X, y, h=0.02, pad=0.25):
    x_min, x_max = X[:, 0].min()-pad, X[:, 0].max()+pad
    y_min, y_max = X[:, 1].min()-pad, X[:, 1].max()+pad
    xvec = np.arange(x_min, x_max, h)
    yvec = np.arange(y_min, y_max, h)
    xx, yy = np.meshgrid(xvec,yvec )
    
    Z = svc.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    plt.contourf(xx, yy, Z, cmap=plt.cm.Paired, alpha=0.2)

    plt.scatter(X[:,0], X[:,1], s=70, c=y, cmap=mpl.cm.Paired)
    # Support vectors indicated in plot by X's
    sv = svc.support_vectors_
    plt.scatter(sv[:,0], sv[:,1], c='k', marker='x', s=100, linewidths=1)
    
    if svc.kernel == 'linear':
        # Get the margin lines 
        w = svc.coef_[0]
        a = -w[0] / w[1]
        yhyperplane = a * xvec - (svc.intercept_[0]) / w[1]
        margin = 1 / np.sqrt(np.sum(svc.coef_ ** 2))
        ymargin_down = yhyperplane+  - np.sqrt(1 + a ** 2) * margin
        ymargin_up = yhyperplane + np.sqrt(1 + a ** 2) * margin
        plt.plot(xvec,ymargin_down, "k--")
        plt.plot(xvec,ymargin_up, "k--")

    
    
    plt.xlim(x_min, x_max)
    plt.ylim(y_min, y_max)
    plt.xlabel('X1')
    plt.ylabel('X2')
    plt.show()
    print('Number of support vectors: ', svc.support_.size)



In [ ]:
#From here on, you can just use this command to plot.
plot_svc(svc, X, y)


## Messing with $C$

Below we have the model fit using several choices for $C$

In [ ]:
svc = SVC(C=10e-2, kernel='linear', )
svc.fit(X, y)
plot_svc(svc, X, y)


In [ ]:
svc = SVC(C=1, kernel='linear', )
svc.fit(X, y)
plot_svc(svc, X, y)


In [ ]:
svc = SVC(C=10e5, kernel='linear', )
svc.fit(X, y)
plot_svc(svc, X, y)


Note that 
- The cost $C$ in class had the property that big $C$ meant small margin 
- This is a DIFFERENT $C$. In this case, big $C$ means small margin. 

They're both serving the same purpose, i.e. to control how tolerant we are to misclassifications. 

The following code uses some nice functions in `sklearn` to do a CV test to determine the best $C$.

In [ ]:
from sklearn.model_selection import GridSearchCV


In [ ]:
# Select the optimal C parameter by cross-validation
C_list = [0.001, 0.01, 0.1, 1, 5, 10, 100]
tuned_parameters = [{'C': C_list}]
clf = GridSearchCV(SVC(kernel='linear'), tuned_parameters, cv=10, scoring='accuracy')
clf.fit(X, y)

&#9989; **<font color=red>Do this:</font>** Use the `clf.cv_results_` function to determine which $C$ give the best score (note there could be ties). Which $C$ did the function choose? 

In [ ]:
# Your code here

&#9989; **<font color=red>Do this:</font>** Load in the `SVM-Data2.csv` data from the folder. Run this same analysis again, namily: 
- Use the `GridSearchCV` function to determine the best choice of $C$.
- Train an individual `SVC` instance with $C$ set to that value.  Then you can draw the resulting model with the `plot_svc` function from earlier. How does the model do?

# Swapping out the kernel 

In [ ]:
data3 = np.loadtxt('SVM-Data3.csv')
X = data3[:,:2]
y = data3[:,2]


plt.scatter(X[:,0], X[:,1], s=70, c=y, cmap=mpl.cm.Paired)
plt.xlabel('X1')
plt.ylabel('X2')

&#9989; **<font color=red>Do this:</font>** Train a SVC using a radial kernel (this is 'rbf' in this code) and with $C=1$, $\gamma = 1$. Use the `plot_svc` function to see what the learned boundary looks like. 

In [ ]:
# Your code here

&#9989; **<font color=red>Do this:</font>** What happens if you increase $C$ to 100? Is this model looking better or worse than what you had before?

In [ ]:
# Your code here #

&#9989; **<font color=red>Do this:</font>** Use the `GridSearchCV` function to determine the best $C$ and $\gamma$ parameters

# Still have time? 

Download the NIST data set from here: https://archive.ics.uci.edu/ml/datasets/optical+recognition+of+handwritten+digits

You just need two files for now, the training set `optdigits.tra` and the testing set `optdigits.tes`.

In [ ]:
X_train = pd.read_csv('optdigits.tra', header=None)
y_train = X_train[64]
X_train = X_train.drop(X_train.columns[64], axis=1)

X_test = pd.read_csv('optdigits.tes', header=None)
y_test = X_test[64]
X_test = X_test.drop(X_test.columns[64], axis=1)

In [ ]:
print(X_train.shape)
print(X_test.shape)

This data set consists of 8x8 images of handwritten digits. Mess around with the value of $i$ below to see other examples

In [ ]:
i = 27
plt.imshow(X_train.values[i].reshape(8,8), cmap="gray") 
plt.show()
print(f'Data point {i} is labeled as {y_train[i]}')

&#9989; **<font color=red>Do this:</font>** Build a classifier to predict the correct digit for a given handwritten digit. As you do this, answer the following questions:
- What choice of kernel does best? 
- What are the optimal choices of parameters for the SVC? 
- How well does your classifier do? Don't forget that quality measures should always use testing data.

In [ ]:
# Your answer here

# Lab Survey

To get credit for today's lab, fill out the following survey before the end of class:

https://forms.gle/hX8GT5FJ2fNMeTo1A

Note this is the same link for every lab, so you will fill this out multiple times this semester.



-----
### Congratulations, we're done!
Written by Dr. Liz Munch, Michigan State University

<a rel="license" href="http://creativecommons.org/licenses/by-nc/4.0/"><img alt="Creative Commons License" style="border-width:0" src="https://i.creativecommons.org/l/by-nc/4.0/88x31.png" /></a><br />This work is licensed under a <a rel="license" href="http://creativecommons.org/licenses/by-nc/4.0/">Creative Commons Attribution-NonCommercial 4.0 International License</a>.